# DoubleML PLR — Monastic Land Controlling for Other Variables

Estimates the **direct effect** of monastic land on rebellion outcomes after
conditioning on other monastic economic variables (tithes, alms), population,
and wealth. By including these as controls in both nuisance models, this
specification isolates the effect of monastic land holding the other channels
constant — complementary to `jn_r_08_dr_total_effect.ipynb`.

**Estimator:** Robinson PLR with 5-fold cross-fitting.
- Treatment nuisance `E[T | X_T]`: OLS (includes population and wealth).
- Outcome nuisance `E[Y | X_Y]`: logistic GLM (includes tithes, alms, population, wealth).
- Final stage: `Y_tilde ~ T_tilde` (OLS).
- Standard errors: Conley spatial HAC (100 km Bartlett kernel).
- Tithes and alms are normalization-matched to the treatment variable.

Outputs tables to `Output/Tables/dr_ctrl_*.tex` and plots to
`Output/Images/Graphs/dr_ctrl_*.png`. Run cells in order.

In [1]:
# Setup: packages, data loading, standardization, parish centroids
pacman::p_load(
  sf, tidyverse, stargazer, dplyr, ggplot2, jsonlite, conflicted
)
conflict_prefer("filter", "dplyr")
conflict_prefer("select", "dplyr")

PROJECT_ROOT <- tryCatch(
  normalizePath(file.path(dirname(rstudioapi::getActiveDocumentContext()$path), "..")),
  error = function(e) {
    cwd <- normalizePath(getwd())
    if (basename(cwd) == "Code") dirname(cwd) else cwd
  }
)
setwd(PROJECT_ROOT)

pretty_dict <- fromJSON("Code/pretty_dict.json")

pdf <- read_sf(dsn = "Data/Processed/northParishFlows.shp")
pdf$terrainTyp <- ifelse(is.na(pdf$terrainTyp), "Other", pdf$terrainTyp)
pdf$uplands    <- ifelse(pdf$terrainTyp == "Uplands",  1, 0)
pdf$lowlands   <- ifelse(pdf$terrainTyp == "Lowlands", 1, 0)

rdf        <- data.frame(pdf)
# Binarize seats: 1 if any rebel gentry seat present, 0 otherwise
rdf$seats  <- ifelse(rdf$seats > 0, 1, 0)

# Parish centroid coordinates (BNG meters) for Conley SEs
parish_xy <- sf::st_coordinates(sf::st_centroid(sf::st_geometry(pdf)))
rdf$.cx   <- parish_xy[, 1]
rdf$.cy   <- parish_xy[, 2]

# Standardize continuous variables (same as existing notebooks)
std_vars <- c(
  "llandOwned", "llo_sk", "llo_arak",
  "lsmLand",   "lbigLand",
  "lsm_sk",    "lbg_sk",
  "lsm_arak",  "lbg_arak",
  "lotherLand", "lownLand",
  "loth_sk",   "lown_sk",
  "loth_arak",  "lown_arak",
  "ltitheOutT", "lti_sk", "lti_arak",
  "lalmsInTot", "lal_sk", "lal_arak",
  "lni_sk",    "lni_arak",
  "lLStax_pc", "lpopC", "distScot", "area", "mean_slope",
  "wet_1535",  "wet_1536"
)
for (v in std_vars) {
  if (v %in% names(rdf)) {
    rdf[[v]] <- scale(rdf[[v]], center = TRUE, scale = TRUE)[, 1]
  }
}

cat("Data loaded:", nrow(rdf), "parishes\n")

[conflicted] Will prefer dplyr::filter over any other package.


[conflicted] Will prefer dplyr::select over any other package.


Data loaded: 1755 parishes


In [2]:
# --- Model 2: Monastic land controlling for other variables -----------------
# Treatment nuisance adds population (lpopC) and wealth (lLStax_pc) to the
# geographic base. Outcome nuisance further adds tithes and alms (normalization-
# matched to the treatment variable). This holds economic channels constant so
# the estimate reflects the *direct* effect of monastic land per se.

# Geographic/proximity base (shared across treatment and outcome nuisance)
x_geo <- c(
  "bigHouse", "smHouse",               # large/small house proximity (20km)
  "distScot", "area", "mean_slope",    # geographic controls
  "uplands", "lowlands"                # terrain type dummies
)

# Treatment nuisance adds socioeconomic controls
x_t_ctrl <- c(x_geo, "lpopC", "lLStax_pc")

# Outcome nuisance base (tithe and alms added per-spec below)
x_y_base <- c(x_geo, "mg_fsnub", "wet_1535", "wet_1536", "lpopC", "lLStax_pc")

# --- Nine treatment × normalization specifications -------------------------
# tithe/alms entries are normalization-matched to the treatment variable
specs <- list(
  total_raw    = list(treat = "llandOwned",  tithe = "ltitheOutT", alms = "lalmsInTot",
                      label = "Total Land (Raw)",                type = "total",   norm = "raw"),
  total_sk     = list(treat = "llo_sk",      tithe = "lti_sk",    alms = "lal_sk",
                      label = "Total Land (per km\u00b2)",        type = "total",   norm = "sk"),
  total_arak   = list(treat = "llo_arak",    tithe = "lti_arak",  alms = "lal_arak",
                      label = "Total Land (per Arable km\u00b2)", type = "total",   norm = "arak"),
  large_raw    = list(treat = "lbigLand",    tithe = "ltitheOutT", alms = "lalmsInTot",
                      label = "Large House Land (Raw)",           type = "large",   norm = "raw"),
  large_sk     = list(treat = "lbg_sk",      tithe = "lti_sk",    alms = "lal_sk",
                      label = "Large House Land (per km\u00b2)",  type = "large",   norm = "sk"),
  large_arak   = list(treat = "lbg_arak",    tithe = "lti_arak",  alms = "lal_arak",
                      label = "Large House Land (per Arable km\u00b2)", type = "large", norm = "arak"),
  offsite_raw  = list(treat = "lotherLand",  tithe = "ltitheOutT", alms = "lalmsInTot",
                      label = "Off-site Land (Raw)",              type = "offsite", norm = "raw"),
  offsite_sk   = list(treat = "loth_sk",     tithe = "lti_sk",    alms = "lal_sk",
                      label = "Off-site Land (per km\u00b2)",     type = "offsite", norm = "sk"),
  offsite_arak = list(treat = "loth_arak",   tithe = "lti_arak",  alms = "lal_arak",
                      label = "Off-site Land (per Arable km\u00b2)", type = "offsite", norm = "arak")
)

outcomes <- c("muster", "primary", "seats")
cat("Defined", length(specs), "treatment specifications,", length(outcomes), "outcomes.\n")

Defined 9 treatment specifications, 3 outcomes.


In [3]:
# --- Helper: DoubleML PLR with K-fold cross-fitting -------------------------
#
# Robinson (1988) / Chernozhukov et al. (2018) PLR:
#   T_tilde = T - E[T | X_T]   (OLS nuisance)
#   Y_tilde = Y - E[Y | X_Y]   (logistic nuisance, response scale)
#   theta = coef of T_tilde in OLS(Y_tilde ~ T_tilde)
#
# Separate covariate sets X_T and X_Y permit different theoretical exclusion
# restrictions: X_T excludes tithes/alms from propensity; X_Y includes them
# to isolate the direct effect of monastic land on rebellion.

run_dr_plr <- function(y_col, t_col, x_t_cols, x_y_cols, data, K = 5, seed = 42) {
  req_cols <- unique(c(y_col, t_col, x_t_cols, x_y_cols))
  cc       <- complete.cases(data[, req_cols, drop = FALSE])
  df       <- data[cc, ]
  n        <- nrow(df)

  set.seed(seed)
  folds   <- sample(rep(seq_len(K), length.out = n))
  T_tilde <- numeric(n)
  Y_tilde <- numeric(n)

  for (k in seq_len(K)) {
    train <- df[folds != k, ]
    test  <- df[folds == k, ]

    # Treatment nuisance: E[T | X_T] via OLS
    m_fit <- lm(
      as.formula(paste(t_col, "~", paste(x_t_cols, collapse = " + "))),
      data = train
    )
    T_tilde[folds == k] <- test[[t_col]] - predict(m_fit, newdata = test)

    # Outcome nuisance: E[Y | X_Y] via logistic regression
    l_fit <- glm(
      as.formula(paste(y_col, "~", paste(x_y_cols, collapse = " + "))),
      data = train, family = binomial(link = "logit")
    )
    Y_tilde[folds == k] <- test[[y_col]] -
      predict(l_fit, newdata = test, type = "response")
  }

  # Final stage: Y_tilde ~ T_tilde (OLS)
  final_df          <- df
  final_df$.Y_tilde <- Y_tilde
  final_df$.T_tilde <- T_tilde
  final_model       <- lm(.Y_tilde ~ .T_tilde, data = final_df)

  list(
    model   = final_model,
    df      = final_df,
    cc_rows = which(cc),
    Y_tilde = Y_tilde,
    T_tilde = T_tilde,
    n       = n,
    K       = K
  )
}

# --- Helper: Conley (spatial HAC) SEs for the final-stage OLS ---------------
# Bartlett kernel at cutoff_km; distances from BNG-meter centroids.

conley_se_dr <- function(plr_result, cx_full, cy_full, cutoff_km = 100) {
  mod  <- plr_result$model
  rows <- plr_result$cc_rows
  X    <- model.matrix(mod)
  u    <- residuals(mod)
  cx   <- cx_full[rows]
  cy   <- cy_full[rows]

  d_km  <- as.matrix(dist(cbind(cx, cy))) / 1000
  K_mat <- pmax(1 - d_km / cutoff_km, 0)   # Bartlett kernel

  bread <- solve(crossprod(X))
  Xu    <- X * u
  meat  <- crossprod(Xu, K_mat %*% Xu)
  V     <- bread %*% meat %*% bread
  setNames(sqrt(diag(V)), colnames(X))
}

cat("Helper functions defined.\n")

Helper functions defined.


In [4]:
# --- Run DR PLR for all 9 specs × 3 outcomes --------------------------------
all_results <- list()  # all_results[[spec_name]][[outcome]] = plr_result
all_conley  <- list()  # all_conley[[spec_name]][[outcome]]  = named SE vector

for (spec_name in names(specs)) {
  spec <- specs[[spec_name]]
  tr   <- spec$treat

  # Outcome nuisance: add normalization-matched tithe and alms to the base
  x_y_cols <- c(spec$tithe, spec$alms, x_y_base)

  cat(sprintf("\n===== DR PLR [%s] — treatment: %s =====\n", spec_name, tr))
  cat(sprintf("  Outcome nuisance: %s + base\n", paste(c(spec$tithe, spec$alms), collapse = ", ")))

  all_results[[spec_name]] <- list()
  all_conley[[spec_name]]  <- list()

  for (y in outcomes) {
    res  <- run_dr_plr(
      y_col    = y,
      t_col    = tr,
      x_t_cols = x_t_ctrl,
      x_y_cols = x_y_cols,
      data     = rdf,
      K        = 5,
      seed     = 42
    )
    se_c <- conley_se_dr(res, cx_full = rdf$.cx, cy_full = rdf$.cy, cutoff_km = 100)

    all_results[[spec_name]][[y]] <- res
    all_conley[[spec_name]][[y]]  <- se_c

    cat(sprintf("  %-10s  n=%d  coef=%.4f  se_conley=%.4f\n",
                y, res$n,
                coef(res$model)[".T_tilde"],
                se_c[".T_tilde"]))
  }

  # Retrieve final-stage models and Conley SEs
  m_muster  <- all_results[[spec_name]][["muster"]]$model
  m_primary <- all_results[[spec_name]][["primary"]]$model
  m_seats   <- all_results[[spec_name]][["seats"]]$model
  se_m <- all_conley[[spec_name]][["muster"]]
  se_p <- all_conley[[spec_name]][["primary"]]
  se_s <- all_conley[[spec_name]][["seats"]]

  treat_label <- unlist(pretty_dict[tr])
  if (is.null(treat_label) || length(treat_label) == 0) treat_label <- tr

  tithe_label <- unlist(pretty_dict[spec$tithe])
  alms_label  <- unlist(pretty_dict[spec$alms])
  if (is.null(tithe_label)) tithe_label <- spec$tithe
  if (is.null(alms_label))  alms_label  <- spec$alms

  # Stargazer table: three outcome columns
  stargazer(
    m_muster, m_primary, m_seats,
    type             = "latex",
    dep.var.labels    = c("Muster", "Primary", "Seats"),
    title            = paste0("DoubleML PLR — Controlled Effect — ", spec$label),
    label            = paste0("tab:dr_ctrl_", spec_name),
    align            = TRUE,
    table.placement  = "H",
    column.labels    = c("Muster", "Primary", "Seats"),
    se               = list(se_m, se_p, se_s),
    covariate.labels = treat_label,
    omit             = "Intercept",
    add.lines        = list(
      c("DR method",           rep("PLR (Robinson)", 3)),
      c("Cross-fitting K",     rep("5", 3)),
      c("Treatment nuisance",  rep("OLS", 3)),
      c("Outcome nuisance",    rep("Logistic", 3)),
      c("Outcome controls",    rep(paste0(tithe_label, ", ", alms_label, ", pop, wealth"), 3)),
      c("Conley SEs (100km)",  rep("Y", 3)),
      c("N",
        as.character(all_results[[spec_name]][["muster"]]$n),
        as.character(all_results[[spec_name]][["primary"]]$n),
        as.character(all_results[[spec_name]][["seats"]]$n))
    ),
    column.sep.width = ".5pt",
    omit.stat        = c("adj.rsq", "f", "ser"),
    out              = paste0("Output/Tables/dr_ctrl_", spec_name, ".tex")
  )
  cat(sprintf("  -> Wrote Output/Tables/dr_ctrl_%s.tex\n", spec_name))
}

cat("\nAll 9 DR PLR controlled-effect specifications complete.\n")


===== DR PLR [total_raw] — treatment: llandOwned =====
  Outcome nuisance: ltitheOutT, lalmsInTot + base
  muster      n=1391  coef=0.0040  se_conley=0.0038
  primary     n=1391  coef=0.0030  se_conley=0.0029
  seats       n=1391  coef=0.0075  se_conley=0.0079

% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Thu, May 14, 2026 - 10:35:22
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{DoubleML PLR — Controlled Effect — Total Land (Raw)} 
  \label{tab:dr_ctrl_total_raw} 
\begin{tabular}{@{\extracolsep{.5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{3}{c}{.Y\_tilde} \\ 
 & \multicolumn{1}{c}{Muster} & \multicolumn{1}{c}{Primary} & \multicolumn{1}{c}{Seats} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)}\\ 
\


All 9 DR PLR controlled-effect specifications complete.


In [5]:
# --- Summary coefficient plots ----------------------------------------------
# One plot per treatment type (total / large / offsite).
# Rows: normalization × outcome; point = DR estimate, bars = 90% Conley CI.

outcome_labels <- c(muster = "Muster", primary = "Primary", seats = "Seats")
norm_labels    <- c(raw = "Raw", sk = "per km\u00b2", arak = "per Arable km\u00b2")

for (ttype in c("total", "large", "offsite")) {
  type_specs <- Filter(function(s) s$type == ttype, specs)

  plot_rows <- list()
  for (spec_name in names(type_specs)) {
    spec <- type_specs[[spec_name]]
    for (y in outcomes) {
      res  <- all_results[[spec_name]][[y]]
      cs   <- summary(res$model)$coefficients
      se_c <- all_conley[[spec_name]][[y]][".T_tilde"]
      if (!".T_tilde" %in% rownames(cs)) next
      est  <- cs[".T_tilde", "Estimate"]
      plot_rows[[length(plot_rows) + 1L]] <- data.frame(
        norm    = spec$norm,
        outcome = outcome_labels[y],
        est     = est,
        lo      = est - 1.645 * se_c,
        hi      = est + 1.645 * se_c,
        p       = cs[".T_tilde", "Pr(>|t|)"],
        n_obs   = res$n
      )
    }
  }
  pdf_plot <- do.call(rbind, plot_rows)
  pdf_plot$significant <- pdf_plot$p < 0.10
  pdf_plot$row_label   <- paste0(norm_labels[pdf_plot$norm], " — ", pdf_plot$outcome)
  pdf_plot$row_label   <- factor(pdf_plot$row_label, levels = rev(unique(pdf_plot$row_label)))

  type_title <- c(total = "Total Monastic Land", large = "Large House Land",
                  offsite = "Off-site Monastic Land")[ttype]

  p <- ggplot(pdf_plot, aes(x = est, y = row_label)) +
    geom_vline(xintercept = 0, linetype = "dashed", color = "gray50") +
    geom_errorbar(aes(xmin = lo, xmax = hi),
                  width = 0.2, color = "gray30", orientation = "y") +
    geom_point(aes(color = significant), size = 3) +
    scale_color_manual(
      values = c("FALSE" = "gray60", "TRUE" = "#0072B2"),
      labels = c("FALSE" = "Not Significant", "TRUE" = "p < 0.10")
    ) +
    labs(
      x     = "DR Coefficient (PLR, Conley SEs 100 km, 90% CI)",
      y     = "",
      color = "Significance",
      title = paste0("DoubleML PLR — Controlled Effect — ", type_title)
    ) +
    theme_minimal() +
    theme(
      axis.text    = element_text(size = 13),
      axis.title   = element_text(size = 13),
      legend.text  = element_text(size = 12),
      legend.title = element_text(size = 12),
      legend.position = "bottom",
      plot.title   = element_text(size = 14)
    )

  fname <- paste0("Output/Images/Graphs/dr_ctrl_", ttype, "_coefs.png")
  ggsave(fname, plot = p, width = 10, height = 7, dpi = 300)
  cat(sprintf("Saved %s\n", fname))
}

cat("Summary plots complete.\n")

Saved Output/Images/Graphs/dr_ctrl_total_coefs.png


Saved Output/Images/Graphs/dr_ctrl_large_coefs.png


Saved Output/Images/Graphs/dr_ctrl_offsite_coefs.png


Summary plots complete.
